# LSTM Model Creation
This nb creates the LSTM model uesd in the CRZ dashboard

## Load in data

In [22]:
import pandas as pd
import numpy as np
import datetime as dt

from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, Model

In [3]:
data = pd.read_csv('../data/processed_data.csv',index_col=None)

## Split data

In [4]:
# 70-15-15 train-val-test split
timestamps = data["toll_10_minute_block"].sort_values().unique()

train_cutoff = timestamps[int(len(timestamps) * 0.70)]
val_cutoff   = timestamps[int(len(timestamps) * 0.85)]

train = data[
    data["toll_10_minute_block"] < train_cutoff
].copy()

val = data[
    (data["toll_10_minute_block"] >= train_cutoff) &
    (data["toll_10_minute_block"] < val_cutoff)
].copy()

test = data[
    data["toll_10_minute_block"] >= val_cutoff
].copy()

In [5]:
# confirm ranges
print(
    f"""Train:
    \n{train['toll_10_minute_block'].min()}
    \nto {train['toll_10_minute_block'].max()}
    \nTotal records: {len(train):,}, {len(train)/len(data):.2%} of total data"""
)

print("\n------------------------------------------------------------\n")

print(
    f"""Validation:
    \n{val['toll_10_minute_block'].min()}
    \nto {val['toll_10_minute_block'].max()}
    \nTotal records: {len(val):,}, {len(val)/len(data):.2%} of total data"""
)

print("\n------------------------------------------------------------\n")

print(
    f"""*Test*:
    \n{test['toll_10_minute_block'].min()}
    \nto {test['toll_10_minute_block'].max()}
    \nTotal records: {len(test):,}, {len(test)/len(data):.2%} of total data"""
)

Train:
    
2025-01-05 00:00:00
    
to 2026-02-25 11:40:00
    
Total records: 719,700, 70.00% of total data

------------------------------------------------------------

Validation:
    
2026-02-25 11:50:00
    
to 2026-05-25 17:50:00
    
Total records: 154,236, 15.00% of total data

------------------------------------------------------------

*Test*:
    
2026-05-25 18:00:00
    
to 2026-08-22 23:50:00
    
Total records: 154,224, 15.00% of total data


## Feature Engineering

In [6]:
# scaling y-target
traffic_scaler = StandardScaler()

# fit to training data
train["traffic_scaled"] = traffic_scaler.fit_transform(
    train[["traffic_volume"]]
)

# transform val/test data based on fitted scaler
val["traffic_scaled"] = traffic_scaler.transform(
    val[["traffic_volume"]]
)

test["traffic_scaled"] = traffic_scaler.transform(
    test[["traffic_volume"]]
)

In [7]:
# select features for modeling
numeric_features = [
    "traffic_scaled",
    "holiday_ind",
    "overnight_ind",
    "dow_sin",
    "dow_cos"
]

categorical_features = [
    "region_id",
    "group_id"
]

# 10 minute steps for a 24h day
SEQ_LENGTH = 144

### Batch windows

In [16]:
def make_group_dataset(group_df, numeric_features, seq_length=144):

    group_df = (
        group_df
        .sort_values("toll_10_minute_block")
        .reset_index(drop=True)
    )

    # Convert Pandas columns to compact NumPy arrays once
    numeric = group_df[numeric_features].to_numpy(dtype=np.float32)

    region = group_df["region_id"].to_numpy(dtype=np.int32)

    group = group_df["group_id"].to_numpy(dtype=np.int32)

    target = group_df["traffic_scaled"].to_numpy(dtype=np.float32)


    # Previous 144 rows of numeric features
    numeric_ds = tf.keras.utils.timeseries_dataset_from_array(
        data=numeric[:-1],
        targets=None,
        sequence_length=seq_length,
        sequence_stride=1,
        shuffle=False,
        batch_size=None
    )


    # Previous 144 region IDs
    region_ds = tf.keras.utils.timeseries_dataset_from_array(
        data=region[:-1],
        targets=None,
        sequence_length=seq_length,
        sequence_stride=1,
        shuffle=False,
        batch_size=None
    )


    # Previous 144 group IDs
    group_ds = tf.keras.utils.timeseries_dataset_from_array(
        data=group[:-1],
        targets=None,
        sequence_length=seq_length,
        sequence_stride=1,
        shuffle=False,
        batch_size=None
    )


    # Target is the NEXT traffic value
    target_ds = tf.data.Dataset.from_tensor_slices(
        target[seq_length:]
    )


    # Combine everything
    ds = tf.data.Dataset.zip(
        (
            numeric_ds,
            region_ds,
            group_ds,
            target_ds
        )
    )

    return ds

In [17]:
train_ds = None

for group_id, group_df in train.groupby("group_id"):

    group_ds = make_group_dataset(
        group_df,
        numeric_features,
        SEQ_LENGTH
    )

    if train_ds is None:
        train_ds = group_ds
    else:
        train_ds = train_ds.concatenate(group_ds)

In [18]:
BATCH_SIZE = 256

train_ds = (
    train_ds
    .shuffle(20_000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [19]:
batch = next(iter(train_ds))

numeric_batch, region_batch, group_batch, y_batch = batch

print("numeric:", numeric_batch.shape)
print("region:", region_batch.shape)
print("group:", group_batch.shape)
print("target:", y_batch.shape)

numeric: (256, 144, 5)
region: (256, 144)
group: (256, 144)
target: (256,)


### Tensorflow model inputs

In [ ]:


numeric_input = layers.Input(
    shape=(SEQ_LENGTH, len(numeric_features)),
    name="numeric"
)

region_input = layers.Input(
    shape=(SEQ_LENGTH,),
    dtype="int32",
    name="region"
)

group_input = layers.Input(
    shape=(SEQ_LENGTH,),
    dtype="int32",
    name="group"
)

In [21]:
# embedding layers for categorical features
n_regions = int(data["region_id"].max()) + 1
n_groups = int(data["group_id"].max()) + 1

In [23]:
region_embedding = layers.Embedding(
    input_dim=n_regions,
    output_dim=3,
    name="region_embedding"
)(region_input)

group_embedding = layers.Embedding(
    input_dim=n_groups,
    output_dim=4,
    name="group_embedding"
)(group_input)

In [28]:
print("Region embedding shape:", region_embedding.shape)
print("Group embedding shape:", group_embedding.shape)

Region embedding shape: (None, 144, 3)
Group embedding shape: (None, 144, 4)


In [29]:
x = layers.Concatenate(axis=-1)([
    numeric_input,
    region_embedding,
    group_embedding
])

In [31]:
x.shape

(None, 144, 12)

## LSTM Layers

In [ ]:
x = layers.LSTM(
    64,
    name="lstm"
)(x)

x.shape

In [35]:
output = layers.Dense(
    1,
    name="traffic_prediction"
)(x)

### LSTM Modeling

In [36]:
model = Model(
    inputs={
        "numeric": numeric_input,
        "region": region_input,
        "group": group_input
    },
    outputs=output
)

In [37]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ region (InputLayer) │ (None, 144)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group (InputLayer)  │ (None, 144)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric             │ (None, 144, 5)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ region_embedding    │ (None, 144, 3)    │         24 │ region[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_embedding     │ (None, 144, 4)    │         52 │ group[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 144, 12)   │          0 │ numeric[0][0],    │
│ (Concatenate)       │                   │            │ region_embedding… │
│                     │                   │            │ group_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     19,712 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ traffic_prediction  │ (None, 1)         │         65 │ lstm[0][0]        │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,853 (77.55 KB)

 Trainable params: 19,853 (77.55 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

### Validate model

In [39]:
val_ds = None

for group_id, group_df in val.groupby("group_id"):

    group_ds = make_group_dataset(
        group_df,
        numeric_features,
        SEQ_LENGTH
    )

    if val_ds is None:
        val_ds = group_ds
    else:
        val_ds = val_ds.concatenate(group_ds)

In [40]:
val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [41]:
batch = next(iter(val_ds))

numeric_batch, region_batch, group_batch, y_batch = batch

print("numeric:", numeric_batch.shape)
print("region:", region_batch.shape)
print("group:", group_batch.shape)
print("target:", y_batch.shape)

numeric: (256, 144, 5)
region: (256, 144)
group: (256, 144)
target: (256,)
